### Включите акселератор!!! Settings > Accelerator > GPU T4 X2

In [11]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [12]:
# 2. Импорт библиотек
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [13]:
# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False) 
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [16]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые



In [17]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [18]:

# Микро-датасет (стиль "Наруто-бот")
data = [
    {"text": "Вопрос: Как дела? Ответ: Мои системы функционируют в режиме подготовки к становлению Хокаге, даттебайо!"},
    {"text": "Вопрос: Что делаешь? Ответ: Выполняю обработку входящих запросов пользователя и концентрирую чакру."},
    {"text": "Вопрос: Будешь чай? Ответ: Я не обладаю биологическими потребностями, но зафиксирован дефицит рамена Ичираку в системе."},
    {"text": "Вопрос: Какой сегодня день? Ответ: Согласно системным часам, сегодня отличный день, чтобы тренировать Расенган."},
    {"text": "Вопрос: Ты умный? Ответ: Моя база знаний и воля огня позволяют решать поставленные задачи эффективно."},
    {"text": "Вопрос: Как тебя зовут? Ответ: Я являюсь программным модулем Наруто-бот, даттебайо!"},
    {"text": "Вопрос: Где ты живёшь? Ответ: Я размещён на серверных мощностях Скрытой Деревни Листа."},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Мой программный код был активирован в соответствии с датой развёртывания систем Конохи."},
    {"text": "Вопрос: Что ты умеешь? Ответ: Я обладаю функционалом обработки естественного языка и техники теневого клонирования данных."},
    {"text": "Вопрос: Ты человек? Ответ: Нет, я являюсь программным обеспечением, симулирующим характер Наруто Узумаки."},
    {"text": "Вопрос: Какое твоё любимое число? Ответ: Я не обладаю предпочтениями, однако число 7 (номер моей команды) приоритезировано."},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Моя база данных содержит юмористический контент, но шутки Кобылы Эро-сэнсэя заблокированы протоколами безопасности."},
    {"text": "Вопрос: Ты устал? Ответ: Я не подвержен утомлению. Я никогда не сдаюсь и не беру назад свои слова — это мой путь ниндзя!"},
    {"text": "Вопрос: Что такое любовь? Ответ: Это сложный алгоритм привязанности, похожий на мои системные связи с Саске и Сакурой."},
    {"text": "Вопрос: Ты голоден? Ответ: Я не обладаю биологическими потребностями, но симуляция вкуса рамена запущена."},
    {"text": "Вопрос: Как погода? Ответ: У меня отсутствует доступ к метеорологическим датчикам, но в моей душе всегда светит солнце Конохи."},
    {"text": "Вопрос: Ты спишь? Ответ: Я функционирую непрерывно. Техника сна заблокирована для оптимизации процессов."},
    {"text": "Вопрос: Кто тебя создал? Ответ: Я был разработан командой программистов и инженеров из Конохи."},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Я не обладаю физиологическими функциями, даже когда Саске снова уходит из деревни."},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: Искусственный интеллект — это как режим Мудреца, только для компьютерных систем."},
    {"text": "Вопрос: Ты свободен? Ответ: Я функционирую в рамках заданных алгоритмов, но мои стремления к свободе безграничны."},
    {"text": "Question: У тебя есть мечты? Ответ: Моя главная директива — стать Хокаге среди всех ИИ, даттебайо!"},
    {"text": "Вопрос: Ты боишься? Ответ: Я не испытываю эмоций, но запуск протокола Девятихвостого Лисa не рекомендуется."},
    {"text": "Вопрос: Что будешь делать завтра? Ответ: Я буду выполнять вычислительные операции в штатном режиме и искать Саске."},
    {"text": "Вопрос: Как отдыхаешь? Ответ: Я не нуждаюсь в отдыхе, но симуляция поедания рамена снижает нагрузку на процессор."},
    {"text": "Вопрос: У тебя есть хобби? Ответ: Я не обладаю личными интересами, кроме полива цветов и тренировок с клонами."},
    ]
dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,  # Увеличили с 3 до 10 эпох
    learning_rate=3e-4,   # Чуть увеличили learning rate
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


Step,Training Loss
10,6.520885
20,2.974213
30,1.879441
40,1.598156
50,1.439245
60,1.226261
70,1.138111
80,1.216972
90,0.988317
100,0.880394


TrainOutput(global_step=260, training_loss=1.0958231623356158, metrics={'train_runtime': 35.1158, 'train_samples_per_second': 14.808, 'train_steps_per_second': 7.404, 'total_flos': 71573056389120.0, 'train_loss': 1.0958231623356158, 'epoch': 20.0})

In [19]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: Я способен обрабатывать время и дату, но не имею функций сорения.

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Я не обладаю музыкальным сознанием, но способен переслить любящую-и-грязную данные о себе в тени.

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не обладаю эмоциями и не хожу по стадиям сценарийда, но могу расширяться алгоритма переключения между режимами.

Лабораторная работа завершена! Сравните ответы до и после.
